# Extract

En esta etapa se extraen los datos originales del dataset de Netflix proporcionado por Kaggle y se cargan en un DataFrame de Pandas, sin realizar modificaciones.

In [3]:
%pip install pandas


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd

# Cargar el dataset
df = pd.read_csv("netflix_titles.csv")

# Mostrar los primeros 5 registros
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


# Transform

### show_id

| Nombre de columna | Posibles riesgos |
|---|---|
| `show_id` | valor vacio, repeticion, no esten en orden, falten algunos|

### type

| Nombre de columna | Posibles riesgos |
|---|---|
| `type` | valor vacio, mal formato, valor sin sentido|

### title

| Nombre de columna | Posibles riesgos |
|---|---|
| `title` | valor null|

### director

| Nombre de columna | Posibles riesgos |
|---|---|
| `director` | valor null, mal formato (comas entre directores) |

### cast

| Nombre de columna | Posibles riesgos |
|---|---|
| `cast` | null, mal formato (comas entre cast) |


### country

| Nombre de columna | Posibles riesgos |
|---|---|
| `country` |null, mal formato (comas entrepaises) |
 

### date_added

| Nombre de columna | Posibles riesgos |
|---|---|
| `date_added` | null, formateo de la fecha|

### release_year

| Nombre de columna | Posibles riesgos |
|---|---|
| `release_year` | null, tipo de dato no numerico |

### rating

| Nombre de columna | Posibles riesgos |
|---|---|
| `rating` | null, formato|

### duration

| Nombre de columna | Posibles riesgos |
|---|---|
| `duration` | null, formato correcto, diferencia entre min y season|

### listed_in

| Nombre de columna | Posibles riesgos |
|---|---|
| `listed_in` | null, formato correcto de lista|

### description

| Nombre de columna | Posibles riesgos |
|---|---|
| `description` |null |

In [5]:
# Información general del dataset
df.info()

# Cantidad de valores nulos por columna
print("\n--- Valores nulos ---")
print(df.isnull().sum())

# Cantidad de valores duplicados
print("\n--- Filas duplicadas ---")
print(df.duplicated().sum())

# Valores únicos de columnas categóricas
print("\n--- Valores únicos: type ---")
print(df["type"].unique())

print("\n--- Valores únicos: rating ---")
print(df["rating"].unique())

# Cantidad de valores únicos en cada columna
print("\n--- Valores únicos por columna ---")
print(df.nunique())

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB

--- Valores nulos ---
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype:

In [6]:
df[df["rating"].isin(["74 min", "84 min", "66 min"])]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
5541,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,"April 4, 2017",2017,74 min,NaN,Movies,"Louis C.K. muses on religion, eternal love, gi..."
5794,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,"September 16, 2016",2010,84 min,NaN,Movies,Emmy-winning comedy writer Louis C.K. brings h...
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,"August 15, 2016",2015,66 min,NaN,Movies,The comic puts his trademark hilarious/thought...


In [7]:
# ============================================================
# TRANSFORM
# ============================================================

df_clean = df.copy()

# ------------------------------------------------------------
# 1. Corregir valores de duration almacenados en rating
# ------------------------------------------------------------
# Algunas filas contienen valores como "74 min", "84 min"
# y "66 min" en la columna rating, mientras que duration
# aparece como nulo.
#
# Estos valores corresponden a la duración de la película,
# por lo que los movemos de rating a duration.

duration_in_rating = df_clean["rating"].str.match(
    r"^\d+\s+min$",
    na=False
)

df_clean.loc[duration_in_rating, "duration"] = (
    df_clean.loc[duration_in_rating, "rating"]
)

df_clean.loc[duration_in_rating, "rating"] = pd.NA


# ------------------------------------------------------------
# 2. Convertir date_added a formato de fecha
# ------------------------------------------------------------
# date_added originalmente está almacenada como texto.
# La convertimos al tipo datetime para facilitar posteriores
# consultas y análisis por fecha.

df_clean["date_added"] = pd.to_datetime(
    df_clean["date_added"],
    errors="coerce"
)


# ------------------------------------------------------------
# 3. Estandarizar valores de texto
# ------------------------------------------------------------
# Eliminamos espacios innecesarios al inicio y al final
# de los valores de las columnas de texto.

text_columns = [
    "show_id",
    "type",
    "title",
    "director",
    "cast",
    "country",
    "rating",
    "duration",
    "listed_in",
    "description"
]

for column in text_columns:
    df_clean[column] = df_clean[column].str.strip()


# ------------------------------------------------------------
# 4. Estandarizar valores nulos
# ------------------------------------------------------------
# Convertimos cadenas vacías en valores nulos para evitar
# tener diferentes representaciones de información faltante.

df_clean[text_columns] = df_clean[text_columns].replace(
    r"^\s*$",
    pd.NA,
    regex=True
)


# ------------------------------------------------------------
# 5. Separar valores múltiples
# ------------------------------------------------------------
# Algunas columnas contienen varios valores dentro de una
# misma celda separados por comas.
#
# Ejemplo:
# "United States, Canada"
#
# Por ahora conservamos estos valores como listas.
# Posteriormente podrán convertirse en tablas relacionadas
# durante el diseño de la base de datos.

for column in ["director", "cast", "country", "listed_in"]:
    df_clean[column] = df_clean[column].apply(
        lambda x: [item.strip() for item in x.split(",")]
        if pd.notna(x) else pd.NA
    )


# ------------------------------------------------------------
# 6. Verificar el resultado de las transformaciones
# ------------------------------------------------------------

print("Dimensiones del dataset:", df_clean.shape)

print("\nTipos de datos:")
print(df_clean.dtypes)

print("\nValores nulos:")
print(df_clean.isnull().sum())

print("\nPrimeros 5 registros transformados:")
display(df_clean.head())

Dimensiones del dataset: (8807, 12)

Tipos de datos:
show_id                    str
type                       str
title                      str
director                object
cast                    object
country                 object
date_added      datetime64[us]
release_year             int64
rating                     str
duration                   str
listed_in               object
description                str
dtype: object

Valores nulos:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        98
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64

Primeros 5 registros transformados:


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,[Kirsten Johnson],<NA>,[United States],2021-09-25,2020,PG-13,90 min,[Documentaries],"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,<NA>,"[Ama Qamata, Khosi Ngema, Gail Mabalane, Thaba...",[South Africa],2021-09-24,2021,TV-MA,2 Seasons,"[International TV Shows, TV Dramas, TV Mysteries]","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,[Julien Leclercq],"[Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nab...",<NA>,2021-09-24,2021,TV-MA,1 Season,"[Crime TV Shows, International TV Shows, TV Ac...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,<NA>,<NA>,<NA>,2021-09-24,2021,TV-MA,1 Season,"[Docuseries, Reality TV]","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,<NA>,"[Mayur More, Jitendra Kumar, Ranjan Raj, Alam ...",[India],2021-09-24,2021,TV-MA,2 Seasons,"[International TV Shows, Romantic TV Shows, TV...",In a city of coaching centers known to train I...


In [8]:
# ============================================================
# VALIDACIÓN DEL DATASET DESPUÉS DE LA TRANSFORMACIÓN
# ============================================================

# 1. Información general
print("=== INFORMACIÓN GENERAL ===")
df_clean.info()

# 2. Valores nulos por columna
print("\n=== VALORES NULOS ===")
print(df_clean.isnull().sum().to_string())

# 3. Filas completamente duplicadas
print("\n=== FILAS DUPLICADAS ===")
print(df_clean.astype(str).duplicated().sum())

# 4. IDs duplicados
print("\n=== show_id DUPLICADOS ===")
print(df_clean["show_id"].duplicated().sum())

# 5. Valores de type
print("\n=== VALORES DE type ===")
print(df_clean["type"].value_counts(dropna=False))

# 6. Valores de rating
print("\n=== VALORES DE rating ===")
print(df_clean["rating"].value_counts(dropna=False).to_string())

# 7. Valores de duration
print("\n=== VALORES DE duration ===")
print(df_clean["duration"].value_counts(dropna=False).head(20).to_string())

# 8. Valores de release_year
print("\n=== release_year ===")
print(df_clean["release_year"].describe())

# 9. Verificar nuevamente si quedaron duraciones dentro de rating
print("\n=== DURACIONES QUE SIGUEN EN rating ===")

remaining_duration_in_rating = df_clean[
    df_clean["rating"].str.match(r"^\d+\s+min$", na=False)
]

print(remaining_duration_in_rating[
    ["show_id", "title", "rating", "duration"]
])

# 10. Revisar valores nulos de date_added
print("\n=== date_added NULOS ===")
print(df_clean[df_clean["date_added"].isna()][
    ["show_id", "title", "date_added"]
])

# 11. Mostrar los primeros registros transformados
print("\n=== PRIMEROS 5 REGISTROS ===")
display(df_clean.head())

=== INFORMACIÓN GENERAL ===
<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       8807 non-null   str           
 1   type          8807 non-null   str           
 2   title         8807 non-null   str           
 3   director      6173 non-null   object        
 4   cast          7982 non-null   object        
 5   country       7976 non-null   object        
 6   date_added    8709 non-null   datetime64[us]
 7   release_year  8807 non-null   int64         
 8   rating        8800 non-null   str           
 9   duration      8807 non-null   str           
 10  listed_in     8807 non-null   object        
 11  description   8807 non-null   str           
dtypes: datetime64[us](1), int64(1), object(4), str(6)
memory usage: 825.8+ KB

=== VALORES NULOS ===
show_id            0
type               0
title              0
director   

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,[Kirsten Johnson],<NA>,[United States],2021-09-25,2020,PG-13,90 min,[Documentaries],"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,<NA>,"[Ama Qamata, Khosi Ngema, Gail Mabalane, Thaba...",[South Africa],2021-09-24,2021,TV-MA,2 Seasons,"[International TV Shows, TV Dramas, TV Mysteries]","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,[Julien Leclercq],"[Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nab...",<NA>,2021-09-24,2021,TV-MA,1 Season,"[Crime TV Shows, International TV Shows, TV Ac...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,<NA>,<NA>,<NA>,2021-09-24,2021,TV-MA,1 Season,"[Docuseries, Reality TV]","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,<NA>,"[Mayur More, Jitendra Kumar, Ranjan Raj, Alam ...",[India],2021-09-24,2021,TV-MA,2 Seasons,"[International TV Shows, Romantic TV Shows, TV...",In a city of coaching centers known to train I...


In [9]:
from collections import Counter

country_counts = Counter(
    country
    for countries in df_clean["country"].dropna()
    for country in countries
)

country_counts_df = pd.DataFrame(
    country_counts.items(),
    columns=["country", "count"]
).sort_values("country")

with pd.option_context("display.max_rows", None):
    display(country_counts_df)

,country,count
28,,7
116,Afghanistan,1
91,Albania,1
40,Algeria,3
76,Angola,1
24,Argentina,91
99,Armenia,1
11,Australia,160
61,Austria,12
110,Azerbaijan,1


In [10]:
%pip install requests


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import requests

response = requests.get(
    "https://countriesnow.space/api/v0.1/countries/positions"
)

print(response.status_code)
print(response.json())

200
{'error': False, 'msg': 'countries and positions retrieved', 'data': [{'name': 'Afghanistan', 'iso2': 'AF', 'long': 65, 'lat': 33}, {'name': 'Albania', 'iso2': 'AL', 'long': 20, 'lat': 41}, {'name': 'Algeria', 'iso2': 'DZ', 'long': 3, 'lat': 28}, {'name': 'AmericanSamoa', 'iso2': 'AS', 'long': -170, 'lat': -14.3333}, {'name': 'Andorra', 'iso2': 'AD', 'long': 1.6, 'lat': 42.5}, {'name': 'Angola', 'iso2': 'AO', 'long': 18.5, 'lat': -12.5}, {'name': 'Anguilla', 'iso2': 'AI', 'long': -63.1667, 'lat': 18.25}, {'name': 'Antarctica', 'iso2': 'AQ', 'long': '0', 'lat': -90}, {'name': 'Antigua and Barbuda', 'iso2': 'AG', 'long': -61.8, 'lat': 17.05}, {'name': 'Argentina', 'iso2': 'AR', 'long': -64, 'lat': -34}, {'name': 'Armenia', 'iso2': 'AM', 'long': 45, 'lat': 40}, {'name': 'Aruba', 'iso2': 'AW', 'long': -69.9667, 'lat': 12.5}, {'name': 'Australia', 'iso2': 'AU', 'long': 133, 'lat': -27}, {'name': 'Austria', 'iso2': 'AT', 'long': 13.3333, 'lat': 47.3333}, {'name': 'Azerbaijan', 'iso2': 'A

In [12]:
countries_api = response.json()['data']

print(len(countries_api))
print(countries_api)

242
[{'name': 'Afghanistan', 'iso2': 'AF', 'long': 65, 'lat': 33}, {'name': 'Albania', 'iso2': 'AL', 'long': 20, 'lat': 41}, {'name': 'Algeria', 'iso2': 'DZ', 'long': 3, 'lat': 28}, {'name': 'AmericanSamoa', 'iso2': 'AS', 'long': -170, 'lat': -14.3333}, {'name': 'Andorra', 'iso2': 'AD', 'long': 1.6, 'lat': 42.5}, {'name': 'Angola', 'iso2': 'AO', 'long': 18.5, 'lat': -12.5}, {'name': 'Anguilla', 'iso2': 'AI', 'long': -63.1667, 'lat': 18.25}, {'name': 'Antarctica', 'iso2': 'AQ', 'long': '0', 'lat': -90}, {'name': 'Antigua and Barbuda', 'iso2': 'AG', 'long': -61.8, 'lat': 17.05}, {'name': 'Argentina', 'iso2': 'AR', 'long': -64, 'lat': -34}, {'name': 'Armenia', 'iso2': 'AM', 'long': 45, 'lat': 40}, {'name': 'Aruba', 'iso2': 'AW', 'long': -69.9667, 'lat': 12.5}, {'name': 'Australia', 'iso2': 'AU', 'long': 133, 'lat': -27}, {'name': 'Austria', 'iso2': 'AT', 'long': 13.3333, 'lat': 47.3333}, {'name': 'Azerbaijan', 'iso2': 'AZ', 'long': 47.5, 'lat': 40.5}, {'name': 'Bahamas', 'iso2': 'BS', 'lo

In [13]:
# Obtener todos los países únicos del dataset de Netflix
netflix_countries = sorted({
    country
    for countries in df_clean["country"].dropna()
    for country in countries
})

# Obtener los nombres de países de la API
api_countries = {
    country["name"]
    for country in countries_api
}

# Países de Netflix que NO aparecen en la API
countries_not_found = sorted(
    set(netflix_countries) - api_countries
)

print("Países encontrados en Netflix:", len(netflix_countries))
print("Países no encontrados en la API:", len(countries_not_found))

for country in countries_not_found:
    print(country)

Países encontrados en Netflix: 123
Países no encontrados en la API: 8

East Germany
Iran
Palestine
Soviet Union
Vatican City
Venezuela
West Germany


In [14]:
%pip install python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import os
from dotenv import load_dotenv

load_dotenv()

TMDB_API_KEY = os.getenv("TMDB_API_KEY")

print("API key cargada:", TMDB_API_KEY is not None)

API key cargada: True


In [16]:
import requests

# Película de prueba
movie_title = "Louis C.K. 2017"

url = "https://api.themoviedb.org/3/search/movie"

params = {
    "api_key": TMDB_API_KEY,
    "query": movie_title
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

data = response.json()

print(data.keys())
print("Resultados encontrados:", len(data["results"]))

if data["results"]:
    print(data["results"][0])

Status code: 200
dict_keys(['page', 'results', 'total_pages', 'total_results'])
Resultados encontrados: 1
{'adult': False, 'backdrop_path': '/1rRoyJJBr2QceI1OMCSODy77mwq.jpg', 'genre_ids': [35, 10770], 'id': 449674, 'title': 'Louis C.K. 2017', 'original_language': 'en', 'original_title': 'Louis C.K. 2017', 'overview': 'Louis C.K. muses on religion, eternal love, giving dogs drugs, email fights, teachers and more in a live performance from Washington, D.C.', 'popularity': 1.153, 'poster_path': '/hUa2qgwOXIgEoMatz5UpmxevgG7.jpg', 'release_date': '2017-04-04', 'softcore': False, 'video': False, 'vote_average': 7.262, 'vote_count': 282}


In [17]:
movie_id = 449674

url = f"https://api.themoviedb.org/3/movie/{movie_id}"

params = {
    "api_key": TMDB_API_KEY,
    "append_to_response": "credits"
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

movie_data = response.json()

print(movie_data.keys())
print(movie_data['credits'].keys())


Status code: 200
dict_keys(['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count', 'credits'])
dict_keys(['cast', 'crew'])


In [18]:
print(movie_data.keys())
print(movie_data['credits'].keys())
print(movie_data['credits']['crew'])


dict_keys(['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count', 'credits'])
dict_keys(['cast', 'crew'])
[{'adult': False, 'gender': 1, 'id': 65916, 'known_for_department': 'Art', 'name': 'Amy Beth Silver', 'original_name': 'Amy Beth Silver', 'popularity': 0.5887, 'profile_path': None, 'credit_id': '63b4a227a9117f05b0dac0d7', 'department': 'Art', 'job': 'Production Design'}, {'adult': False, 'gender': 2, 'id': 52849, 'known_for_department': 'Acting', 'name': 'Louis C.K.', 'original_name': 'Louis C.K.', 'popularity': 3.0536, 'profile_path': '/1ii770zdeX62UFGawTmNr2vddYb.jpg', 'credit_id': '58e40127c3a3684a7901513e', 'department': 'Writing', 'job': 'Writer'}, {'a

In [19]:
# Ver países de producción
print("=== COUNTRIES ===")
print(movie_data["production_countries"])


# Ver créditos
print("\n=== CREDITS KEYS ===")
print(movie_data["credits"].keys())


# Ver primeros miembros del cast
print("\n=== CAST ===")
print(movie_data["credits"]["cast"][:3])


# Ver miembros del crew relacionados con dirección
print("\n=== DIRECTORS ===")
directors_tmdb = []
for person in movie_data['credits']['crew']:
    if person['job'] == 'Director':
        directors_tmdb.append(person['job'])

print(directors_tmdb)

=== COUNTRIES ===
[{'iso_3166_1': 'US', 'name': 'United States of America'}]

=== CREDITS KEYS ===
dict_keys(['cast', 'crew'])

=== CAST ===
[{'adult': False, 'gender': 2, 'id': 52849, 'known_for_department': 'Acting', 'name': 'Louis C.K.', 'original_name': 'Louis C.K.', 'popularity': 3.0536, 'profile_path': '/1ii770zdeX62UFGawTmNr2vddYb.jpg', 'cast_id': 0, 'character': 'Self', 'credit_id': '58dcdb3fc3a36822ae00934b', 'order': 0}]

=== DIRECTORS ===
['Director']


In [20]:
import requests
import pandas as pd


def get_tmdb_data(row):
    """
    Consulta TMDB usando el título y tipo de contenido.
    Retorna director, cast y country encontrados.
    """

    title = row["title"]
    content_type = row["type"]

    # Buscar título en TMDB
    search_endpoint = (
        "https://api.themoviedb.org/3/search/movie"
        if content_type == "Movie"
        else "https://api.themoviedb.org/3/search/tv"
    )

    params = {
        "api_key": TMDB_API_KEY,
        "query": title
    }

    response = requests.get(search_endpoint, params=params)

    if response.status_code != 200:
        return {
            "director": None,
            "cast": None,
            "country": None
        }

    results = response.json().get("results", [])

    # Si no encuentra el título
    if not results:
        return {
            "director": None,
            "cast": None,
            "country": None
        }

    tmdb_id = results[0]["id"]

    # Obtener detalles
    details_endpoint = (
        f"https://api.themoviedb.org/3/movie/{tmdb_id}"
        if content_type == "Movie"
        else f"https://api.themoviedb.org/3/tv/{tmdb_id}"
    )

    details_params = {
        "api_key": TMDB_API_KEY,
        "append_to_response": "credits"
    }

    details_response = requests.get(
        details_endpoint,
        params=details_params
    )

    if details_response.status_code != 200:
        return {
            "director": None,
            "cast": None,
            "country": None
        }

    data = details_response.json()

    # Obtener directores
    directors = [
        person["name"]
        for person in data.get("credits", {}).get("crew", [])
        if person.get("job") == "Director"
    ]

    # Obtener actores principales
    cast = [
        person["name"]
        for person in data.get("credits", {}).get("cast", [])[:10]
    ]

    # Obtener países
    countries = [
        country["name"]
        for country in data.get("production_countries", [])
    ]

    return {
        "director": directors if directors else None,
        "cast": cast if cast else None,
        "country": countries if countries else None
    }

In [21]:
test_row = df_clean.iloc[5541]

result = get_tmdb_data(test_row)

result

{'director': ['Louis C.K.'],
 'cast': ['Louis C.K.'],
 'country': ['United States of America']}

In [22]:
# Seleccionar registros donde falta algún dato que queremos completar
missing_data_sample = df_clean[
    df_clean["director"].isna() |
    df_clean["cast"].isna() |
    df_clean["country"].isna()
].head(5)

display(missing_data_sample[["title", "type", "director", "cast", "country"]])

,title,type,director,cast,country
0,Dick Johnson Is Dead,Movie,[Kirsten Johnson],<NA>,[United States]
1,Blood & Water,TV Show,<NA>,"[Ama Qamata, Khosi Ngema, Gail Mabalane, Thaba...",[South Africa]
2,Ganglands,TV Show,[Julien Leclercq],"[Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nab...",<NA>
3,Jailbirds New Orleans,TV Show,<NA>,<NA>,<NA>
4,Kota Factory,TV Show,<NA>,"[Mayur More, Jitendra Kumar, Ranjan Raj, Alam ...",[India]


In [23]:
for index, row in missing_data_sample.iterrows():
    print("=" * 60)
    print("Título:", row["title"])
    print("Tipo:", row["type"])

    tmdb_result = get_tmdb_data(row)

    print(tmdb_result)

Título: Dick Johnson Is Dead
Tipo: Movie
{'director': ['Kirsten Johnson'], 'cast': ['Richard Johnson', 'Kirsten Johnson', 'Isla Sierck', 'Jed Sierck', 'Felix Torres', 'Viva Torres', 'Raymond Damazo', 'Michael Hilow', 'Michael Melton', 'Joanne Reiner Tucker'], 'country': ['United States of America']}
Título: Blood & Water
Tipo: TV Show
{'director': None, 'cast': ['Ama Qamata', 'Khosi Ngema', 'Gail Mabalane', 'Dillon Windvogel', 'Thabang Molaba', 'Arno Greeff', 'Natasha Thahane', 'Mekaila Mathys', 'Greteli Fincham', 'Leroy Panashe Siyafa'], 'country': ['South Africa']}
Título: Ganglands
Tipo: TV Show
{'director': ['Julien Leclercq'], 'cast': ['Sami Bouajila', 'Tracy Gotoas', 'Salim Kéchiouche', 'Samuel Jouy', 'Aron Kto', 'Lola Le Lann'], 'country': ['Belgium', 'France']}
Título: Jailbirds New Orleans
Tipo: TV Show
{'director': None, 'cast': None, 'country': ['United States of America']}
Título: Kota Factory
Tipo: TV Show
{'director': ['Pratish Mehta', 'Raghav Subbu'], 'cast': ['Jitendra 

In [24]:
# Tomar registros que tienen país en Netflix
sample_countries = df_clean[
    df_clean["country"].notna()
].head(100)

tmdb_country_names = set()

for index, row in sample_countries.iterrows():

    result = get_tmdb_data(row)

    if result["country"]:
        for country in result["country"]:
            tmdb_country_names.add(country)


print("Países encontrados en TMDB:")
for country in sorted(tmdb_country_names):
    print(country)

Países encontrados en TMDB:
Argentina
Australia
Belgium
Burkina Faso
Canada
China
Czech Republic
Ethiopia
Finland
France
Germany
Ghana
Hong Kong
India
Italy
Japan
Mexico
Netherlands
Nigeria
Romania
Russia
Singapore
South Africa
South Korea
Spain
Thailand
Turkey
United Kingdom
United States of America
Venezuela


In [25]:
missing_country = df_clean[df_clean["country"].isna()].copy()

print("Registros sin país:", len(missing_country))

Registros sin país: 831


In [26]:
import requests
import time

session = requests.Session()

def get_tmdb_country(title, content_type):
    
    # Endpoint de búsqueda
    if content_type == "Movie":
        search_url = "https://api.themoviedb.org/3/search/movie"
    else:
        search_url = "https://api.themoviedb.org/3/search/tv"

    params = {
        "api_key": TMDB_API_KEY,
        "query": title
    }

    try:
        response = session.get(
            search_url,
            params=params,
            timeout=10
        )

        response.raise_for_status()

        results = response.json().get("results", [])

        # No se encontró el título
        if not results:
            return None

        # Tomamos el primer resultado
        tmdb_id = results[0]["id"]

        # Endpoint de detalles
        if content_type == "Movie":
            details_url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
        else:
            details_url = f"https://api.themoviedb.org/3/tv/{tmdb_id}"

        response = session.get(
            details_url,
            params={"api_key": TMDB_API_KEY},
            timeout=10
        )

        response.raise_for_status()

        data = response.json()

        countries = data.get("production_countries", [])

        return [
            country["name"]
            for country in countries
        ]

    except requests.exceptions.RequestException:
        return None

In [27]:
for index, row in missing_country.iterrows():

    countries = get_tmdb_country(
        row["title"],
        row["type"]
    )

    if countries:
        df_clean.at[index, "country"] = countries

    print(
        f"{index}: {row['title']} -> {countries}"
    )

    time.sleep(0.05)

2: Ganglands -> ['Belgium', 'France']
3: Jailbirds New Orleans -> ['United States of America']
5: Midnight Mass -> ['United States of America']
6: My Little Pony: A New Generation -> ['Canada', 'Ireland']
10: Vendetta: Truth, Lies and The Mafia -> ['Italy', 'United Kingdom']
11: Bangkok Breaking -> ['Thailand']
13: Confessions of an Invisible Girl -> ['Brazil']
14: Crime Stories: India Detectives -> ['United Kingdom']
16: Europe's Most Dangerous Man: Otto Skorzeny in Spain -> ['Spain']
18: Intrusion -> ['United States of America']
19: Jaguar -> ['Spain']
20: Monsters Inside: The 24 Faces of Billy Milligan -> []
22: Avvai Shanmughi -> None
23: Go! Go! Cory Carson: Chrissy Takes the Wheel -> ['United States of America']
26: Minsara Kanavu -> ['India']
30: Ankahi Kahaniya -> ['India']
31: Chicago Party Aunt -> ['United States of America']
33: Squid Game -> ['South Korea']
34: Tayo and Little Wizards -> ['South Korea']
35: The Father Who Moves Mountains -> ['Romania', 'Sweden']
36: The Str

In [28]:
print("============================================")
print("      AUDITORÍA FINAL DEL DATASET")
print("============================================")

# 1. Información general
print("\n=== INFORMACIÓN GENERAL ===")
df_clean.info()

# 2. Valores nulos
print("\n=== VALORES NULOS ===")
print(df_clean.isna().sum())

# 3. Porcentaje de nulos
print("\n=== PORCENTAJE DE NULOS ===")
print(
    (df_clean.isna().mean() * 100)
    .round(2)
    .sort_values(ascending=False)
)

# ============================================================
# DUPLICADOS
# ============================================================

print("\n=== FILAS DUPLICADAS ===")

# Convertimos temporalmente las listas a texto para poder
# comprobar duplicados de filas completas.
df_duplicates_check = df_clean.copy()

for column in ["director", "cast", "country", "listed_in"]:
    df_duplicates_check[column] = df_duplicates_check[column].apply(
        lambda x: tuple(x) if isinstance(x, list) else x
    )

print(
    "Filas completamente duplicadas:",
    df_duplicates_check.duplicated().sum()
)


# ============================================================
# DUPLICADOS POR show_id
# ============================================================

print("\n=== show_id DUPLICADOS ===")

duplicated_ids = df_clean[
    df_clean["show_id"].duplicated(keep=False)
]

print("show_id duplicados:", len(duplicated_ids))

if len(duplicated_ids) > 0:
    display(
        duplicated_ids[
            ["show_id", "title", "type"]
        ].sort_values("show_id")
    )
else:
    print("No existen show_id duplicados.")

# 6. Valores de type
print("\n=== TYPE ===")
print(df_clean["type"].value_counts(dropna=False))

# 7. Ratings
print("\n=== RATING ===")
print(df_clean["rating"].value_counts(dropna=False))

# 8. Duraciones
print("\n=== DURATION ===")
print(df_clean["duration"].value_counts(dropna=False).head(30))

# 9. Tipos de datos
print("\n=== TIPOS DE DATOS ===")
print(df_clean.dtypes)

      AUDITORÍA FINAL DEL DATASET

=== INFORMACIÓN GENERAL ===
<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       8807 non-null   str           
 1   type          8807 non-null   str           
 2   title         8807 non-null   str           
 3   director      6173 non-null   object        
 4   cast          7982 non-null   object        
 5   country       8557 non-null   object        
 6   date_added    8709 non-null   datetime64[us]
 7   release_year  8807 non-null   int64         
 8   rating        8800 non-null   str           
 9   duration      8807 non-null   str           
 10  listed_in     8807 non-null   object        
 11  description   8807 non-null   str           
dtypes: datetime64[us](1), int64(1), object(4), str(6)
memory usage: 825.8+ KB

=== VALORES NULOS ===
show_id            0
type              

In [29]:
# ============================================================
# COMPLETAR DIRECTORES FALTANTES CON TMDB
# USANDO TÍTULO + RELEASE YEAR
# ============================================================

movies_missing_director = df_clean[
    (df_clean["type"] == "Movie") &
    (df_clean["director"].isna())
].copy()

print(
    "Películas sin director antes de TMDB:",
    len(movies_missing_director)
)

directors_found = 0
directors_not_found = 0

for index, row in movies_missing_director.iterrows():

    result = get_tmdb_data(row)

    if result["director"]:

        df_clean.at[index, "director"] = result["director"]

        directors_found += 1

    else:
        directors_not_found += 1


print("\n============================================")
print("RESULTADO")
print("============================================")

print("Directores encontrados:", directors_found)
print("Directores no encontrados:", directors_not_found)

print(
    "Películas sin director después de TMDB:",
    df_clean[
        (df_clean["type"] == "Movie") &
        (df_clean["director"].isna())
    ].shape[0]
)

Películas sin director antes de TMDB: 188

RESULTADO
Directores encontrados: 115
Directores no encontrados: 73
Películas sin director después de TMDB: 73


In [30]:
# ============================================================
# PREPARAR date_added PARA MYSQL
# ============================================================

# Asegurar que la columna sea datetime
df_clean["date_added"] = pd.to_datetime(
    df_clean["date_added"],
    errors="coerce"
)

# Como MySQL utilizará DATE, eliminamos la parte de hora
df_clean["date_added"] = df_clean["date_added"].dt.date

print("Tipo de dato de date_added:")
print(df_clean["date_added"].dtype)

print("\nPrimeras fechas:")
print(df_clean["date_added"].head())

print("\nFechas nulas:")
print(df_clean["date_added"].isna().sum())

Tipo de dato de date_added:
object

Primeras fechas:
0    2021-09-25
1    2021-09-24
2    2021-09-24
3    2021-09-24
4    2021-09-24
Name: date_added, dtype: object

Fechas nulas:
98


In [31]:
# Verificar que las fechas tienen el formato esperado
display(
    df_clean[["show_id", "title", "date_added"]].head(20)
)

,show_id,title,date_added
0,s1,Dick Johnson Is Dead,2021-09-25
1,s2,Blood & Water,2021-09-24
2,s3,Ganglands,2021-09-24
3,s4,Jailbirds New Orleans,2021-09-24
4,s5,Kota Factory,2021-09-24
5,s6,Midnight Mass,2021-09-24
6,s7,My Little Pony: A New Generation,2021-09-24
7,s8,Sankofa,2021-09-24
8,s9,The Great British Baking Show,2021-09-24
9,s10,The Starling,2021-09-24
